# Assignment_2: Q1

## 1. Setup

In [54]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 15)

RAW_PATH = Path("raw_dataset") / "train.csv"
PROCESSED_DIR = Path("processed_dataset")
OUTPUT_DIR = Path("output")

print("pandas version:", pd.__version__)

pandas version: 3.0.5


## 2. Load and check the raw data

In [55]:
raw_cars = pd.read_csv(RAW_PATH, index_col=0)

print("shape:", raw_cars.shape)

raw_cars.head()

shape: (5847, 13)


,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [56]:
raw_cars.info()

<class 'pandas.DataFrame'>
Index: 5847 entries, 1 to 6018
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   str    
 1   Location           5847 non-null   str    
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   str    
 5   Transmission       5847 non-null   str    
 6   Owner_Type         5847 non-null   str    
 7   Mileage            5845 non-null   str    
 8   Engine             5811 non-null   str    
 9   Power              5811 non-null   str    
 10  Seats              5809 non-null   float64
 11  New_Price          815 non-null    str    
 12  Price              5847 non-null   float64
dtypes: float64(2), int64(2), str(9)
memory usage: 639.5 KB


## 3. Task (a)

### Check the missing values and their types

In [57]:
missing_report = pd.DataFrame({
    "missing_count": raw_cars.isna().sum(),
    "missing_percent": (raw_cars.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

missing_report

,missing_count,missing_percent
New_Price,5032,86.06
Seats,38,0.65
Engine,36,0.62
Power,36,0.62
Mileage,2,0.03
Fuel_Type,0,0.00
Kilometers_Driven,0,0.00
Year,0,0.00
Location,0,0.00
Name,0,0.00


In [58]:
raw_cars.loc[
    raw_cars["Mileage"].isna(),
    ["Name", "Fuel_Type", "Mileage", "Engine", "Power"],]

,Name,Fuel_Type,Mileage,Engine,Power
4446,Mahindra E Verito D4,Electric,NaN,72 CC,41 bhp
4904,Toyota Prius 2009-2016 Z4,Electric,NaN,1798 CC,73 bhp


In [59]:
engine_missing = raw_cars["Engine"].isna()

print("Rows missing Engine:", int(engine_missing.sum()))
print("also missing Power:", int((engine_missing & raw_cars["Power"].isna()).sum()))
print("also missing Seats:", int((engine_missing & raw_cars["Seats"].isna()).sum()))
print("Rows missing Seats only:", int((~engine_missing & raw_cars["Seats"].isna()).sum()))

Rows missing Engine: 36
also missing Power: 36
also missing Seats: 36
Rows missing Seats only: 2


### Based on the results for missing values shown above:
1. In cloumn `New_price`, there are 5032 missing values (86.06%), so I will drop this cloumn becasue if I fill in 86% of this column with some numbers, most of the data in that column would be fictitious, which would cover the actual price differences between the various models.
2. In cloumn `Mileage`, there are 2 missing values (0.03%), I will drop these two rows becasue both of these vehicles are electric cars. For this type of vehicle, there are no fuel efficiency figures expressed in kilometers per liter, so any values calculated based on them are meaningless.
3. In cloumn `Engine`, `Power`, there are 36 missing values (0.62%), I will drop these rows because these 36 rows are missing both of these specifications, as well as 36 of the 38 missing `Seats` values. Since engine displacement and power vary significantly across different vehicle models, calculating the median by column could result in pairing the engines of luxury SUVs with those of hatchbacks, leading to even greater errors. Therefore, deleting 0.6% of the data is a more feasible solution.
4. In cloumn `Seats`, there are 2 missing values (0.03%), I will replace them with mode, becasue after the dropping in the previous step, only two vehicles are left with missing seat counts. Since `Seats` is a discrete variable and there is a category with a high probability, I will insert mode value. The number of seats must be an integer, so the mean (5.28) would result in fractional seat counts that cannot exist in the data. The mode (5) is the simplest valid choice.

In [60]:
cars = raw_cars.drop(columns="New_Price")

print("columns after drop:", cars.shape[1])

columns after drop: 12


In [61]:
rows_before = len(cars)
cars = cars.dropna(subset=["Mileage", "Engine", "Power"])

print("Rows before:", rows_before)
print("Rows after: ", len(cars))
print("Rows dropped:", rows_before - len(cars))

Rows before: 5847
Rows after:  5809
Rows dropped: 38


In [62]:
cars.loc[cars["Seats"].isna(), ["Name", "Location", "Year", "Seats"]]

,Name,Location,Year,Seats
229,Ford Figo Diesel,Bangalore,2015,NaN
1917,Honda City 1.5 EXI,Jaipur,2005,NaN


In [63]:
cars["Seats"] = cars["Seats"].fillna(cars["Seats"].mode()[0])

### Double check

In [64]:
print("Remaining missing values:", int(cars.isna().sum().sum()))
print("Rows retained:", len(cars), f"({len(cars) / len(raw_cars):.2%} of the raw data)")

Remaining missing values: 0
Rows retained: 5809 (99.35% of the raw data)


## 4. Task (b)

### Remove the units

In [65]:
for column in ["Mileage", "Engine", "Power"]:
    print(f"{column:8} units: {cars[column].str.split().str[1].unique().tolist()}")
print(f"{'New_Price':8} units: {raw_cars['New_Price'].str.split().str[1].dropna().unique().tolist()}") 

Mileage  units: ['kmpl', 'km/kg']
Engine   units: ['CC']
Power    units: ['bhp']
New_Price units: ['Lakh', 'Cr']


In [66]:
cars["Mileage"] = cars["Mileage"].str.split().str[0].astype(float)
cars["Engine"] = cars["Engine"].str.split().str[0].astype(int)
cars["Power"] = cars["Power"].str.split().str[0].astype(float)

In [67]:
new_price_parts = raw_cars["New_Price"].dropna().str.split(expand=True)
new_price_lakh = (
    new_price_parts[0].astype(float) * new_price_parts[1].map({"Lakh": 1, "Cr": 100})
)

pd.DataFrame({
    "New_Price (raw)": raw_cars.loc[new_price_lakh.index, "New_Price"],
    "New_Price (lakh)": new_price_lakh,
}).sort_values("New_Price (lakh)", ascending=False).head()

,New_Price (raw),New_Price (lakh)
4079,2.3 Cr,230.0
4778,1.71 Cr,171.0
3199,1.66 Cr,166.0
3752,1.6 Cr,160.0
2056,1.4 Cr,140.0


In [68]:
display(cars[["Mileage", "Engine", "Power"]].dtypes)
cars[["Name", "Mileage", "Engine", "Power"]].head()

Mileage    float64
Engine       int64
Power      float64
dtype: object

,Name,Mileage,Engine,Power
1,Hyundai Creta 1.6 CRDi SX Option,19.67,1582,126.20
2,Honda Jazz V,13.00,1199,88.70
3,Maruti Ertiga VDI,20.77,1248,88.76
4,Audi A4 New 2.0 TDI Multitronic,15.20,1968,140.80
6,Nissan Micra Diesel XV,23.08,1461,63.10


## 5. Task (c)

### Numerical one hot encoded value

In [69]:
display(cars["Fuel_Type"].value_counts())
display(cars["Transmission"].value_counts())

Fuel_Type
Diesel    3153
Petrol    2656
Name: count, dtype: int64

Transmission
Manual       4107
Automatic    1702
Name: count, dtype: int64

In [70]:
cars = pd.get_dummies(
    cars,
    columns=["Fuel_Type", "Transmission"],
    dtype=int,
)

encoded_columns = [
    column for column in cars.columns
    if column.startswith(("Fuel_Type_", "Transmission_"))
]

cars[["Name"] + encoded_columns].head()

,Name,Fuel_Type_Diesel,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
1,Hyundai Creta 1.6 CRDi SX Option,1,0,0,1
2,Honda Jazz V,0,1,0,1
3,Maruti Ertiga VDI,1,0,0,1
4,Audi A4 New 2.0 TDI Multitronic,1,0,1,0
6,Nissan Micra Diesel XV,1,0,0,1


## 6. Task (d)

### Create one more feature about Car Age

In [71]:
CURRENT_YEAR = date.today().year

cars["Car_Age"] = CURRENT_YEAR - cars["Year"]

print("Current year:", CURRENT_YEAR)
cars[["Name", "Year", "Car_Age"]].head()

Current year: 2026


,Name,Year,Car_Age
1,Hyundai Creta 1.6 CRDi SX Option,2015,11
2,Honda Jazz V,2011,15
3,Maruti Ertiga VDI,2012,14
4,Audi A4 New 2.0 TDI Multitronic,2013,13
6,Nissan Micra Diesel XV,2013,13


In [72]:
cars["Car_Age"].describe()

count    5809.000000
mean       12.525392
std         3.172185
min         7.000000
25%        10.000000
50%        12.000000
75%        14.000000
max        28.000000
Name: Car_Age, dtype: float64

## 7. Task (e)

### Select, filter, rename, mutate, arrange, and summarize with group by city

In [73]:
selected = cars[[
    "Name", "Location", "Year", "Car_Age",
    "Kilometers_Driven", "Price",
]]

selected.head()

,Name,Location,Year,Car_Age,Kilometers_Driven,Price
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,11,41000,12.50
2,Honda Jazz V,Chennai,2011,15,46000,4.50
3,Maruti Ertiga VDI,Chennai,2012,14,87000,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,13,40670,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,13,86999,3.50


In [74]:
recent_automatics = selected.loc[(selected["Price"] <= 10) & (selected["Car_Age"] <= 10)]

print("Rows before filter:", len(selected))
print("Rows after filter: ", len(recent_automatics))
recent_automatics.head()

Rows before filter: 5809
Rows after filter:  1112


,Name,Location,Year,Car_Age,Kilometers_Driven,Price
10,Maruti Ciaz Zeta,Kochi,2018,8,25692,9.95
16,Honda Amaze S i-Dtech,Kochi,2016,10,58950,5.40
17,Maruti Swift DDiS VDI,Jaipur,2017,9,25000,5.99
28,Honda WRV i-VTEC VX,Kochi,2018,8,37430,9.90
36,Maruti Vitara Brezza ZDi Plus,Kochi,2018,8,50075,8.63


In [75]:
renamed = recent_automatics.rename(columns={
    "Kilometers_Driven": "KM",
    "Price": "PRICE",
})

renamed.head()

,Name,Location,Year,Car_Age,KM,PRICE
10,Maruti Ciaz Zeta,Kochi,2018,8,25692,9.95
16,Honda Amaze S i-Dtech,Kochi,2016,10,58950,5.40
17,Maruti Swift DDiS VDI,Jaipur,2017,9,25000,5.99
28,Honda WRV i-VTEC VX,Kochi,2018,8,37430,9.90
36,Maruti Vitara Brezza ZDi Plus,Kochi,2018,8,50075,8.63


In [76]:
mutated = renamed.assign(Km_per_Year=(renamed["KM"] / renamed["Car_Age"]).round(0),)

mutated.head()

,Name,Location,Year,Car_Age,KM,PRICE,Km_per_Year
10,Maruti Ciaz Zeta,Kochi,2018,8,25692,9.95,3212.0
16,Honda Amaze S i-Dtech,Kochi,2016,10,58950,5.40,5895.0
17,Maruti Swift DDiS VDI,Jaipur,2017,9,25000,5.99,2778.0
28,Honda WRV i-VTEC VX,Kochi,2018,8,37430,9.90,4679.0
36,Maruti Vitara Brezza ZDi Plus,Kochi,2018,8,50075,8.63,6259.0


In [77]:
arranged = mutated.sort_values("PRICE", ascending=False)

arranged.head(10)

,Name,Location,Year,Car_Age,KM,PRICE,Km_per_Year
2204,Mahindra Scorpio S10 AT 4WD,Pune,2017,9,66988,10.00,7443.0
269,Renault Duster 110PS Diesel RXZ Option,Delhi,2016,10,26000,10.00,2600.0
5679,Volkswagen Vento 1.5 TDI Highline,Hyderabad,2017,9,28000,10.00,3111.0
4255,Renault Duster 110PS Diesel RxL AMT,Bangalore,2016,10,26000,10.00,2600.0
3141,Tata Safari Storme VX,Jaipur,2016,10,36000,10.00,3600.0
4907,Honda City i-VTEC VX,Ahmedabad,2017,9,29000,10.00,3222.0
5799,Volkswagen Vento 1.5 TDI Highline AT,Pune,2016,10,19000,10.00,1900.0
3719,Ford EcoSport 1.5 Petrol Titanium Plus AT,Pune,2017,9,14000,9.99,1556.0
1905,Maruti Ciaz 1.4 Zeta,Kochi,2018,8,15982,9.96,1998.0
5903,Maruti Ciaz 1.4 AT Alpha,Hyderabad,2017,9,15234,9.95,1693.0


In [78]:
location_summary = (
    cars.groupby("Location", as_index=False)
    .agg(
        listings=("Name", "count"),
        average_price=("Price", "mean"),
        median_price=("Price", "median"),
        average_car_age=("Car_Age", "mean"),
        automatic_share=("Transmission_Automatic", "mean"),
    )
    .round(2)
    .sort_values("median_price", ascending=False)
)

location_summary

,Location,listings,average_price,median_price,average_car_age,automatic_share
3,Coimbatore,629,15.14,8.36,10.55,0.39
1,Bangalore,348,13.61,7.10,13.10,0.41
7,Kochi,637,11.32,6.82,10.47,0.32
0,Ahmedabad,217,8.59,6.00,12.63,0.27
5,Hyderabad,709,10.00,5.99,13.17,0.27
9,Mumbai,757,9.57,5.90,12.61,0.39
4,Delhi,535,9.91,5.70,12.62,0.32
2,Chennai,474,7.96,4.72,13.92,0.25
10,Pune,581,6.99,4.50,13.54,0.24
6,Jaipur,401,5.94,4.25,13.34,0.13


### 8. Export

In [79]:
processed_path = PROCESSED_DIR / "used_cars_processed.csv"
cars.to_csv(processed_path)

print("Exported:", processed_path.resolve())

Exported: D:\Assignment_2\Q1\processed_dataset\used_cars_processed.csv


In [80]:
summary_path = OUTPUT_DIR / "price_summary_by_city.csv"
location_summary.to_csv(summary_path, index=False)

print("Exported:", summary_path.resolve())

Exported: D:\Assignment_2\Q1\output\price_summary_by_city.csv
